# Performance Analysis

Analysis of load testing metrics from Ollama inference endpoint.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)


In [ ]:
# Load metrics
df = pd.read_csv("metrics.csv")
df.head()

## Latency Distribution by Prompt Type and Concurrency

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# TTFT distribution
successful = df[df["success"] == True]
sns.boxplot(data=successful, x="batch_label", y="ttft_ms", ax=axes[0])
axes[0].set_title("Time to First Token (TTFT)")
axes[0].set_ylabel("TTFT (ms)")
axes[0].tick_params(axis="x", rotation=45)

# Total latency distribution
sns.boxplot(data=successful, x="batch_label", y="total_time_ms", ax=axes[1])
axes[1].set_title("Total Request Latency")
axes[1].set_ylabel("Latency (ms)")
axes[1].tick_params(axis="x", rotation=45)

plt.tight_layout()
plt.savefig("latency_distribution.png", dpi=150, bbox_inches="tight")
plt.show()

## Throughput: Tokens per Second

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
tps = successful[successful["tokens_per_sec"] > 0]
sns.barplot(data=tps, x="batch_label", y="tokens_per_sec", errorbar="sd", ax=ax)
ax.set_title("Tokens per Second by Configuration")
ax.set_ylabel("Tokens/sec")
ax.tick_params(axis="x", rotation=45)
plt.tight_layout()
plt.savefig("throughput.png", dpi=150, bbox_inches="tight")
plt.show()

## Percentile Summary Table

In [ ]:
summary = successful.groupby("batch_label").agg(
    count=("total_time_ms", "count"),
    ttft_p50=("ttft_ms", lambda x: np.percentile(x, 50)),
    ttft_p95=("ttft_ms", lambda x: np.percentile(x, 95)),
    ttft_p99=("ttft_ms", lambda x: np.percentile(x, 99)),
    latency_p50=("total_time_ms", lambda x: np.percentile(x, 50)),
    latency_p95=("total_time_ms", lambda x: np.percentile(x, 95)),
    latency_p99=("total_time_ms", lambda x: np.percentile(x, 99)),
    tps_mean=("tokens_per_sec", "mean"),
).round(1)

summary

## Commentary

**Key Findings:**

1. **TTFT scales linearly with concurrency** since Ollama processes requests sequentially by default. Higher concurrency means requests queue.

2. **Long prompts have higher TTFT** due to prompt evaluation (prefill) time scaling with input length.

3. **Tokens/sec is relatively stable** across prompt lengths because generation speed is primarily bound by model architecture, not input size.

4. **Stop sequences reduce total latency** by terminating generation early, which is useful for evaluation tasks with short expected answers.

5. **P95/P99 tail latencies** are significantly higher than P50 under concurrency, indicating queuing effects.

**Production Implications:**
- For evaluation pipelines, sequential processing (concurrency=1) gives the most predictable latency
- Batch processing with stop sequences is the most efficient for benchmarks with short answers
- For scaling, consider vLLM with continuous batching instead of Ollama
